# Topic modeling with BERTopic

In [ ]:
import sqlite3
import pandas as pd
from tqdm import tqdm

c:\Users\Zofia\Desktop\z\płaca\nokia\dblp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Prepare data

In [ ]:
db_path = "../data/processed/dblp.db"

conn = sqlite3.connect(db_path)

query = """
SELECT id, title, year, venue, type
FROM papers
WHERE title IS NOT NULL
"""

df = pd.read_sql_query(query, conn)

conn.close()

df.head()

,id,title,year,venue,type
0,series/sapere/Freed13,Practical Introspection as Inspiration for AI.,2011,PT-AI,inproceedings
1,series/sapere/Steiner13,C.S. Peirce and Artificial Intelligence: Histo...,2011,PT-AI,inproceedings
2,series/sapere/Sandberg13,Feasibility of Whole Brain Emulation.,2011,PT-AI,inproceedings
3,series/sapere/BerkeleyR13,Machine Mentality?,2011,PT-AI,inproceedings
4,series/sapere/NasutoB13,Of (Zombie) Mice and Animats.,2011,PT-AI,inproceedings


In [ ]:
df_sample = df.sample(n=50000, random_state=42)
docs = df_sample["title"].tolist()

## BERTopic

In [27]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

### Parameteres

In [29]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(
    n_neighbors=50,
    n_components=5,
    min_dist = 0.2,
    metric="cosine"
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom"
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=20
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3717.51it/s]


### Train

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)


2026-04-25 20:59:26,792 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 1563/1563 [29:29<00:00,  1.13s/it] 
2026-04-25 21:28:57,022 - BERTopic - Embedding - Completed ✓
2026-04-25 21:28:57,022 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-25 21:29:17,602 - BERTopic - Dimensionality - Completed ✓
2026-04-25 21:29:17,604 - BERTopic - Cluster - Start clustering the reduced embeddings


AttributeError: No prediction data was generated

In [ ]:
topic_model.reduce_topics(docs, nr_topics=100)

2026-04-25 19:42:39,629 - BERTopic - Topic reduction - Reducing number of topics
2026-04-25 19:42:39,780 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 19:42:40,260 - BERTopic - Representation - Completed ✓
2026-04-25 19:42:40,270 - BERTopic - Topic reduction - Reduced number of topics from 615 to 5


### Results

In [15]:
# Topic list
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,23156,-1_of_for_and_in,"[of, for, and, in, the, on, based, with, to, u...",[Energy-Efficient and Robust In-Network Infere...
1,0,23785,0_for_of_and_in,"[for, of, and, in, the, based, on, with, learn...",[The Impact of Cross-Validation Schemes for EE...
2,1,1758,1_of_for_with_and,"[of, for, with, and, control, the, robot, syst...","[Minimal information based, simple identificat..."
3,2,1246,2_of_power_for_and,"[of, power, for, and, with, optical, in, based...",[An analysis of energy storage system interact...
4,3,55,3_holographic_lattice_boltzmann_plasma,"[holographic, lattice, boltzmann, plasma, part...",[Hydrodynamic optimization of channelling devi...


In [16]:
# Words for specific topic
topic_model.get_topic(0)

[('for', np.float64(0.06841520087667263)),
 ('of', np.float64(0.060193890331387896)),
 ('and', np.float64(0.05792808504423596)),
 ('in', np.float64(0.051084960589759476)),
 ('the', np.float64(0.04236752104737433)),
 ('based', np.float64(0.04169544126936414)),
 ('on', np.float64(0.0379436911033352)),
 ('with', np.float64(0.034767246675244654)),
 ('learning', np.float64(0.030165989887023553)),
 ('using', np.float64(0.028060949474971394))]

In [17]:
# Topic visualization
topic_model.visualize_topics()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[0, 'for | of | and | in | the', 23785],
                                   [1, 'of | for | with | and | control', 1758],
                                   [2, 'of | power | for | and | with', 1246],
                                   [3, 'holographic | lattice | boltzmann | plasma | particle', 55]],
                                  dtype=object),
              'hovertemplate': '<b>Topic %{customdata[0]}</b><br>%{customdata[1]}<br>Size: %{customdata[2]}',
              'legendgroup': '',
              'marker': {'color': '#B0BEC5',
                         'line': {'color': 'DarkSlateGrey', 'width': 2},
                         'size': {'bdata': '6VzeBt4ENwA=', 'dtype': 'i2'},
                         'sizemode': 'area',
                         'sizeref': 14.865625,
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': '8oJYwQSmXMFiOU7BXBcywQ==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'f3I8QVkhMEFM6i1BBCdEQQ==', 'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'annotations': [{'showarrow': False,
                                'text': 'D1',
                                'x': np.float32(-15.859111),
                                'y': np.float32(11.66885),
                                'yshift': 10},
                               {'showarrow': False,
                                'text': 'D2',
                                'x': np.float32(-12.660105),
                                'xshift': 10,
                                'y': np.float32(14.098454)}],
               'height': 650,
               'hoverlabel': {'bgcolor': 'white', 'font': {'family': 'Rockwell', 'size': 16}},
               'legend': {'itemsizing': 'constant', 'tracegroupgap': 0},
               'margin': {'t': 60},
               'shapes': [{'line': {'color': '#CFD8DC', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(-12.660105),
                           'x1': np.float32(-12.660105),
                           'y0': np.float32(9.239246),
                           'y1': np.float32(14.098454)},
                          {'line': {'color': '#9E9E9E', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(-15.859111),
                           'x1': np.float32(-9.461098),
                           'y0': np.float32(11.66885),
                           'y1': np.float32(11.66885)}],
               'sliders': [{'active': 0,
                            'pad': {'t': 50},
                            'steps': [{'args': [{'marker.color': [['red', '#B0BEC5', '#B0BEC5', '#B0BEC5']]}],
                                       'label': 'Topic 0',
                                       'method': 'update'},
                                      {'args': [{'marker.color': [['#B0BEC5', 'red', '#B0BEC5', '#B0BEC5']]}],
                                       'label': 'Topic 1',
                                       'method': 'update'},
                                      {'args': [{'marker.color': [['#B0BEC5', '#B0BEC5', 'red', '#B0BEC5']]}],
                                       'label': 'Topic 2',
                                       'method': 'update'},
                                      {'args': [{'marker.color': [['#B0BEC5', '#B0BEC5', '#B0BEC5', 'red']]}],
                                       'label': 'Topic 3',
                                       'method': 'update'}]}],
               'template': '...',
               'title': {'font': {'color': 'Black', 'size': 22},
                         'text': '<b>Intertopic Distance Map</b>',
                         'x': 0.5,
                         'xanchor': 'center',
                         'y': 0.95,
                         'yanch

In [18]:
# Topic relations
topic_model.visualize_hierarchy()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'marker': {'color': 'rgb(61,153,112)'},
              'mode': 'lines',
              'type': 'scatter',
              'x': {'bdata': 'AAAAAAAAAABsYLqNUUHSP2xguo1RQdI/AAAAAAAAAAA=', 'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAALsAAAAAAAAAuwAAAAAAAADnAAAAAAAAAOcA=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hoverinfo': 'text',
              'marker': {'color': 'rgb(61,153,112)'},
              'mode': 'lines',
              'type': 'scatter',
              'x': {'bdata': 'bGC6jVFB0j9Gy+vEyibaP0bL68TKJto/AAAAAAAAAAA=', 'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAANMAAAAAAAAA0wAAAAAAAgEHAAAAAAACAQcA=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hoverinfo': 'text',
              'marker': {'color': 'rgb(61,153,112)'},
              'mode': 'lines',
              'type': 'scatter',
              'x': {'bdata': 'AAAAAAAAAABls84rvyToP2Wzziu/JOg/RsvrxMom2j8=', 'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAAFMAAAAAAAAAUwAAAAAAAgDvAAAAAAACAO8A=', 'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'autosize': False,
               'height': 260,
               'hoverlabel': {'bgcolor': 'white', 'font': {'family': 'Rockwell', 'size': 16}},
               'hovermode': 'closest',
               'plot_bgcolor': '#ECEFF1',
               'showlegend': False,
               'template': '...',
               'title': {'font': {'color': 'Black', 'size': 22},
                         'text': '<b>Hierarchical Clustering</b>',
                         'x': 0.5,
                         'xanchor': 'center',
                         'yanchor': 'top'},
               'width': 1000,
               'xaxis': {'mirror': 'allticks',
                         'rangemode': 'tozero',
                         'showgrid': False,
                         'showline': True,
                         'showticklabels': True,
                         'ticks': 'outside',
                         'type': 'linear',
                         'zeroline': False},
               'yaxis': {'mirror': 'allticks',
                         'range': [-40.0, 0.0],
                         'rangemode': 'tozero',
                         'showgrid': False,
                         'showline': True,
                         'showticklabels': True,
                         'tickmode': 'array',
                         'ticks': 'outside',
                         'ticktext': [3_holographic_lattice_boltz...,
                                      1_of_for_with, 0_for_of_and, 2_of_power_for],
                         'tickvals': [-5.0, -15.0, -25.0, -35.0],
                         'type': 'linear',
                         'zeroline': False}}
})

In [19]:
# Most important words
topic_model.visualize_barchart(top_n_topics=10)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': '#D55E00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.04236752104737433, 0.051084960589759476,
                    0.05792808504423596, 0.060193890331387896, 0.06841520087667263],
              'xaxis': 'x',
              'y': [the  , in  , and  , of  , for  ],
              'yaxis': 'y'},
             {'marker': {'color': '#0072B2'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.052821203381041404, 0.05369380386210803,
                    0.05673597493654042, 0.07579008065884056, 0.08455482293599847],
              'xaxis': 'x2',
              'y': [control  , and  , with  , for  , of  ],
              'yaxis': 'y2'},
             {'marker': {'color': '#CC79A7'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.04604105766146343, 0.050490058375773215,
                    0.06282612367018832, 0.06318926708922813, 0.06659930930434362],
              'xaxis': 'x3',
              'y': [with  , and  , for  , power  , of  ],
              'yaxis': 'y3'},
             {'marker': {'color': '#E69F00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.09124204805102099, 0.10991656427071533, 0.1324180319743965,
                    0.1408783906581865, 0.1485760126170127],
              'xaxis': 'x4',
              'y': [particle  , plasma  , boltzmann  , lattice  , holographic  ],
              'yaxis': 'y4'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Topic 0',
                                'x': 0.0875,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Topic 1',
                                'x': 0.36250000000000004,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Topic 2',
                                'x': 0.6375000000000001,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Topic 3',
                                'x': 0.9125,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'}],
               'height': 325.0,
               'hoverlabel': {'bgcolor': 'white', 'font': {'family': 'Rockwell', 'size': 16}},
               'showlegend': False,
               'template': '...',
               'title': {'font': {'color': 'Black', 'size': 22},
                         'text': 'Topic Word Scores',
                         'x': 0.5,
                         'xanchor': 'center',
                         'yanchor': 'top'},
               'width': 1000,
               'xaxis': {'anchor': 'y', 'domain': [0.0, 0.175], 'showgrid': True},
               'xaxis2': {'anchor': 'y2', 'domain': [0.275, 0.45], 'showgrid': True},
               'xax

In [20]:
df_sample["topic"] = topics

In [23]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df_sample["year"].tolist()
)

topic_model.visualize_topics_over_time(topics_over_time)

16it [00:02,  7.55it/s]


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'hovertext': [<b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: of, for, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, the,
                            <b>Topic 0</b><br>Words: for, of, and, in, based,
                            <b>Topic 0</b><br>Words: for, of, and, in, based,
                            <b>Topic 0</b><br>Words: for, and, of, in, based,
                            <b>Topic 0</b><br>Words: for, and, of, in, based,
                            <b>Topic 0</b><br>Words: for, and, of, in, based],
              'marker': {'color': '#E69F00'},
              'mode': 'lines',
              'name': '0_for_of_and_in',
              'type': 'scatter',
              'x': {'bdata': '2gfbB9wH3QfeB98H4AfhB+IH4wfkB+UH5gfnB+gH6Qc=', 'dtype': 'i2'},
              'y': {'bdata': 'bAM8A8ID7AMhBEkElwT2BF4FeQbtBmoHYwd8CFMJPAk=', 'dtype': 'i2'}},
             {'hoverinfo': 'text',
              'hovertext': [<b>Topic 1</b><br>Words: of, for, with, robot, the,
                            <b>Topic 1</b><br>Words: of, for, the, control,
                            equations, <b>Topic 1</b><br>Words: for, of, the, with,
                            equations, <b>Topic 1</b><br>Words: of, for, and, the,
                            with, <b>Topic 1</b><br>Words: of, for, robot, the,
                            with, <b>Topic 1</b><br>Words: of, for, the, robot,
                            with, <b>Topic 1</b><br>Words: of, for, control, the,
                            robot, <b>Topic 1</b><br>Words: of, for, robot, with,
                            control, <b>Topic 1</b><br>Words: of, for, with, and,
                            the, <b>Topic 1</b><br>Words: of, for, with, and,
                            control, <b>Topic 1</b><br>Words: of, for, with, and,
                            control, <b>Topic 1</b><br>Words: of, for, control,
                            with, the, <b>Topic 1</b><br>Words: of, for, and, with,
                            the, <b>Topic 1</b><br>Words: of, for, with, robot,
                            the, <b>Topic 1</b><br>Words: of, for, and, with,
                            control, <b>Topic 1</b><br>Words: for, of, robot, and,
                            with],
              'marker': {'color': '#56B4E9'},
              'mode': 'lines',
              'name': '1_of_for_with_and',
              'type': 'scatter',
              'x': {'bdata': '2gfbB9wH3QfeB98H4AfhB+IH4wfkB+UH5gfnB+gH6Qc=', 'dtype': 'i2'},
              'y': {'bdata': 'MwBIAFMATgBQAEwAXwBbAGoAoQCBAI8AlQCSAKEAiQA=', 'dtype': 'i2'}},
             {'hoverinfo': 'text',
              'hovertext': [<b>Topic 2</b><br>Words: of, power, for, and, in,
                            <b>Topic 2</b><br>Words: of, for, and, grid, with,
                            <b>Topic 2</b><br>Words: power, for, of, in, grid,
                            <b>Topic 2</b><br>Words: power, of, with, and, for,
                            <b>Topic 2</b><br>Words: of, power, and, grid, with,
                            <b>Topic 2</b><br>Words: of, for, power, optical, and,
                            <b>Topic 2</b><br>Words: of, for, power, optical, and,
                            <b>Topic 2</b><br>Words: of, for, power, with, and,
         

In [24]:
topic_model.save('bertopic_dblp_model')

2026-04-25 19:50:21,529 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
